In [10]:
import numpy as np
import scipy.stats as scs
from pyvbmc import VBMC

In [11]:
D = 2  # We consider a 2-D problem

def log_likelihood(theta):
    """D-dimensional Rosenbrock's banana function."""
    theta = np.atleast_2d(theta)

    x, y = theta[:, :-1], theta[:, 1:]
    return -np.sum((x**2 - y) ** 2 + (x - 1) ** 2 / 100, axis=1)

prior_mu = np.zeros((1, D))
prior_std = 3 * np.ones((1, D))

def log_prior(x):
    """Independent normal prior."""
    return np.sum(scs.norm.logpdf(x, prior_mu, prior_std))
def log_joint(x):
    """log-density of the joint distribution."""
    return log_likelihood(x) + log_prior(x)

LB = np.full((1, D), -np.inf)  # Lower bounds
UB = np.full((1, D), np.inf)  # Upper bounds
PLB = prior_mu - prior_std  # Plausible lower bounds
PUB = prior_mu + prior_std  # Plausible upper bounds
x0 = np.copy(prior_mu)

In [13]:
initial_samples = np.random.uniform(PLB, PUB, size=(100 * D, D))
initial_eval = np.array([log_joint(x) for x in initial_samples])

In [14]:
print(D, initial_samples.shape)

2 (200, 2)


In [15]:
from pyvbmc import VBMC, VariationalPosterior

In [20]:
options = {
    "f_vals": initial_eval,
    # "sample_count": initial_samples.shape[0],
    # "sample_count": 100 * D,
    # "X_init": initial_samples,
}
vbmc = VBMC(log_joint, initial_samples, LB, UB, PLB, PUB, options=options)


In [21]:
vp, results = vbmc.optimize()

Beginning variational optimization assuming EXACT observations of the log-joint.
 Iteration f-count/f-cache    Mean[ELBO]     Std[ELBO]     sKL-iter[q]   K[q]  Convergence    Action
More than sample_count=10 initial points have been provided, using only the first 10 points.
     0         0  /   10          -3.48          1.43      51845.78        2        inf       start warm-up
     1         5  /   10          -1.79          1.31         28.55        2        inf       
     2        10  /   10          -2.84          0.02          1.44        2       37.4       
     3        15  /   10          -2.80          0.00          0.12        2          3       
     4        20  /   10          -2.79          0.00          0.00        2     0.0989       end warm-up


KeyboardInterrupt: 

In [32]:
# First, generate a large number of samples from the variational posterior:
n_samples = int(3e5)
Xs, _ = vp.sample(n_samples)

# Easily compute statistics such as moments, credible intervals, etc.
post_mean = np.mean(Xs, axis=0)  # Posterior mean
post_cov = np.cov(Xs.T)  # Posterior covariance matrix
print("The approximate posterior mean is:", post_mean)
print("The approximate posterior covariance matrix is:\n", post_cov)

The approximate posterior mean is: [0.04375488 1.11516072]
The approximate posterior covariance matrix is:
 [[1.176104   0.11346305]
 [0.11346305 1.83658264]]


In [62]:
import numpy as np
import scipy.stats as scs
from scipy.optimize import minimize
from pyvbmc import VBMC
from pyvbmc.formatting import format_dict

D = 4  # A four-dimensional problem
prior_mu = np.zeros(D)
prior_var = 3 * np.ones(D)

def log_prior(theta):
    """Multivariate normal prior on theta."""
    cov = np.diag(prior_var)
    return scs.multivariate_normal(prior_mu, cov).logpdf(theta)

def log_likelihood(theta, data=np.ones(D)):
    """D-dimensional Rosenbrock's banana function."""
    # In this simple demo the data just translates the parameters:
    theta = np.atleast_2d(theta)
    theta = theta + data

    x, y = theta[:, :-1], theta[:, 1:]
    return -np.sum((x**2 - y) ** 2 + (x - 1) ** 2 / 100, axis=1)

def log_joint(theta, data=np.ones(D)):
    """log-density of the joint distribution."""
    return log_likelihood(theta, data) + log_prior(theta)

LB = np.full(D, -np.inf)  # Lower bounds
UB = np.full(D, np.inf)  # Upper bounds
PLB = np.full(D, prior_mu - np.sqrt(prior_var))  # Plausible lower bounds
PUB = np.full(D, prior_mu + np.sqrt(prior_var))  # Plausible upper bounds

In [63]:
np.random.seed(41)
x0 = np.random.uniform(PLB, PUB)  # Random point inside plausible box
x0 = minimize(
    lambda t: -log_joint(t),
    x0,
    bounds=[
        (-np.inf, np.inf),
        (-np.inf, np.inf),
        (-np.inf, np.inf),
        (-np.inf, np.inf),
    ],
).x
np.random.seed(42)

In [ ]:
# Limit number of function evaluations
options = {
    "max_fun_evals": 2,
    "max_iter": 2,
}
# We can specify either the log-joint, or the log-likelihood and log-prior.
# In other words, the following lines are equivalent:
vbmc = VBMC(
    log_likelihood, x0, LB, UB, PLB, PUB,
    options=options,
    log_prior=log_prior,
)
vp, results = vbmc.optimize()

Reshaping x0 to row vector.
Reshaping lower bounds to (1, 4).
Reshaping upper bounds to (1, 4).
Reshaping plausible lower bounds to (1, 4).
Reshaping plausible upper bounds to (1, 4).
Beginning variational optimization assuming EXACT observations of the log-joint.
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
     0         10          -1.99         1.05    349207.27        2        inf     start warm-up
     1         15          -3.35         1.10         1.08        2        inf     
     2         20          -4.01         1.23         0.34        2         12     
     3         25          -4.05         1.11        12.43        2        211     
     4         30          -5.44         0.41         1.94        2       38.3     
   inf         30          -5.16         0.37         1.59       50       38.3     finalize
Inference terminated: reached maximum number of function evaluations options.max_fun_evals.
Estimated ELBO: -5.163 +/-0.

In [66]:
vp, results = vbmc.optimize()

Beginning variational optimization assuming EXACT observations of the log-joint.
Continuing optimization from previous state.
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
     5         35          -5.32         0.27         0.25        2       5.38     
   inf         35          -4.87         0.51         0.79       50       5.38     finalize
Inference terminated: reached maximum number of function evaluations options.max_fun_evals.
Estimated ELBO: -4.869 +/-0.511.
Caution: Returned variational solution may have not converged.


In [67]:
vp, results = vbmc.optimize()

Beginning variational optimization assuming EXACT observations of the log-joint.
Continuing optimization from previous state.
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
     6         40          -5.26         0.17         0.16        2       3.45     
   inf         40          -4.99         0.19         0.17       50       3.45     finalize
Inference terminated: reached maximum number of function evaluations options.max_fun_evals.
Estimated ELBO: -4.988 +/-0.191.
Caution: Returned variational solution may have not converged.


In [ ]:
print(format_dict(results))
vbmc.save("vbmc_test_save.pkl", overwrite=True)

{
    'function': '<function VBMC._init_log_joint.<locals>.log_joint at 0x7d3f44152d40>',
    'problem_type': 'unconstrained',
    'iterations': 6,
    'func_count': 40,
    'best_iter': np.int64(5),
    'train_set_size': np.int64(35),
    'components': 50,
    'r_index': np.float64(8.847043373346827),
    'convergence_status': 'no',
    'overhead': nan,
    'rng_state': 'rng',
    'algorithm': 'Variational Bayesian Monte Carlo',
    'version': '1.0.4',
    'message': 'Inference terminated: reached maximum number of function evaluations options.max_fun_evals.',
    'elbo': np.float64(-4.684720507673049),
    'elbo_sd': np.float64(0.4092575375137467),
    'success_flag': False,
}


In [ ]:
new_options = {
    "max_fun_evals": 50 * (D + 2),
}
vbmc = VBMC.load(
    "vbmc_test_save.pkl",
    new_options=new_options,
    iteration=None,  # the default: start from the last stored iteration.
    set_random_state=False,  # the default: don't modify the random state
    # (can be set to True for reproducibility).
)
vp, results = vbmc.optimize()

Beginning variational optimization assuming EXACT observations of the log-joint.
Continuing optimization from previous state.
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
     7         45          -4.70         0.10         0.01        2      0.617     
     8         50          -4.61         0.09         0.02        2      0.946     
     9         55          -4.63         0.05         0.01        2      0.388     end warm-up
    10         60          -4.60         0.01         0.02        2      0.394     
    11         65          -4.57         0.01         0.00        2      0.174     
    12         70          -4.50         0.01         0.04        5      0.883     
    13         75          -4.43         0.00         0.02        8      0.495     rotoscale, undo rotoscale
    14         80          -4.34         0.00         0.02       11      0.598     
    15         85          -4.26         0.00         0.02       13      0.5

In [51]:
vbmc.save("vbmc_test_save.pkl", overwrite=True)
vbmc = VBMC.load("vbmc_test_save.pkl")

samples, components = vbmc.vp.sample(5)
# `samples` are samples drawn from the variational posterior.
# `components` are the index of the mixture components each
#  sample was drawn from.
print(samples)
print(components)

[[-1.31612049 -1.52730187 -0.39147328  0.25310104]
 [-1.18309519 -0.34958821 -0.32727893  1.07563724]
 [-1.91034387  0.40565372  0.59373051  1.60024278]
 [-0.38797566 -0.2896129  -0.86825919 -0.4468939 ]
 [-1.13307058 -1.36498086 -1.06991662 -0.12817713]]
[ 8  5 14  0  3]


In [52]:
import numpy as np
import scipy.stats as scs
from pyvbmc import VBMC, VariationalPosterior
import matplotlib.pyplot as plt

In [ ]:
D = 2  # We'll use a 2-D problem, again for speed
prior_mu = np.zeros(D)
prior_var = 3 * np.ones(D)
LB = np.full((1, D), -np.inf)  # Lower bounds
UB = np.full((1, D), np.inf)  # Upper bounds
PLB = np.full((1, D), prior_mu - np.sqrt(prior_var))  # Plausible lower bounds
PUB = np.full((1, D), prior_mu + np.sqrt(prior_var))  # Plausible upper


def log_prior(theta):
    """Multivariate normal prior on theta, same as before."""
    cov = np.diag(prior_var)
    return scs.multivariate_normal(prior_mu, cov).logpdf(theta)

# log-likelihood (Rosenbrock)
def log_likelihood(theta):
    """D-dimensional Rosenbrock's banana function."""
    theta = np.atleast_2d(theta)
    n, D = theta.shape

    # Standard deviation of synthetic noise:
    noise_sd = np.sqrt(1.0 + 0.5 * np.linalg.norm(theta) ** 2)

    # Rosenbrock likelihood:
    x, y = theta[:, :-1], theta[:, 1:]
    base_density = -np.sum((x**2 - y) ** 2 + (x - 1) ** 2 / 100, axis=1)

    noisy_estimate = base_density + noise_sd * np.random.normal(size=(n, 1))
    return noisy_estimate, noise_sd

# Full model:
def log_joint(theta, data=np.ones(D)):
    """log-density of the joint distribution."""
    log_p = log_prior(theta)
    log_l, noise_est = log_likelihood(theta)
    # For the joint, we have to add log densities and carry-through the noise estimate.
    return log_p + log_l, noise_est
x0 = np.zeros((1, D))  # Initial point

In [60]:
options = {"specify_target_noise": True}
vbmc = VBMC(
    log_joint, x0,
    LB, UB, PLB, PUB,
    options=options,
)
np.random.seed(42)
vp, results = vbmc.optimize()

Beginning variational optimization assuming NOISY observations of the log-joint
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
     0         10          -1.09         0.67    254274.53        2        inf     start warm-up
     1         15          -2.84         0.51         3.00        2        inf     
     2         20          -2.40         0.53         0.12        2       3.75     
     3         25          -2.66         0.38         0.16        2       4.33     
     4         30          -2.34         0.35         0.01        2      0.771     end warm-up
     5         35          -2.27         0.34         0.07        2       1.93     
     6         40          -2.30         0.29         0.03        2       1.04     
     7         45          -2.24         0.26         0.01        3      0.438     
     8         50          -2.31         0.25         0.01        6      0.593     
     9         55          -2.20         0.24     